# Projeto Final Aplicado a Ciência de Dados
## Deteção de Outliers em Contratos Públicos e Previsão do Montante de Adjudição

## Grupo 08
## Membros:
###          - Afonso Alexandre nº 123425;
###          - André  Freitas nº 123409;
###          - Gonçalo Vilhena nº 123384;
###          - Rodolfo Nardy nº 123419




In [1]:
import pandas as pd
import os
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

In [2]:
file_path = "contratos_total.csv"
df = pd.read_csv(file_path, low_memory=False)

data_dir = "./dados/"
arquivo = "contratos2015.xlsx"
df_2015= pd.read_excel(data_dir + arquivo)

#df = pd.read_parquet("contratos_total.parquet")

df.head()
df.info()
df.shape
df.columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2046213 entries, 0 to 2046212
Data columns (total 35 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   idcontrato                int64  
 1   nAnuncio                  object 
 2   TipoAnuncio               object 
 3   idINCM                    float64
 4   tipoContrato              object 
 5   idprocedimento            int64  
 6   tipoprocedimento          object 
 7   objectoContrato           object 
 8   descContrato              object 
 9   adjudicante               object 
 10  adjudicatarios            object 
 11  dataPublicacao            object 
 12  dataCelebracaoContrato    object 
 13  precoContratual           float64
 14  CPV                       object 
 15  prazoExecucao             int64  
 16  LocalExecucao             object 
 17  fundamentacao             object 
 18  ProcedimentoCentralizado  object 
 19  numAcordoQuadro           float64
 20  DescrAcordoQuadro       

Index(['idcontrato', 'nAnuncio', 'TipoAnuncio', 'idINCM', 'tipoContrato',
       'idprocedimento', 'tipoprocedimento', 'objectoContrato', 'descContrato',
       'adjudicante', 'adjudicatarios', 'dataPublicacao',
       'dataCelebracaoContrato', 'precoContratual', 'CPV', 'prazoExecucao',
       'LocalExecucao', 'fundamentacao', 'ProcedimentoCentralizado',
       'numAcordoQuadro', 'DescrAcordoQuadro', 'precoBaseProcedimento',
       'dataDecisaoAdjudicacao', 'dataFechoContrato', 'PrecoTotalEfetivo',
       'regime', 'justifNReducEscrContrato', 'tipoFimContrato',
       'CritMateriais', 'concorrentes', 'linkPecasProc', 'Observacoes',
       'ContratEcologico', 'fundamentAjusteDireto', 'Ano'],
      dtype='object')

In [7]:
(df['LocalExecucao'].str.strip().str.lower() == "portugal").sum()

np.int64(143138)

In [3]:
percent_nulos = (df.isnull().sum() / len(df)) * 100

# Ordem crescente
percent_nulos_ordenado = percent_nulos.sort_values(ascending=False)
print(percent_nulos_ordenado)

Observacoes                 96.582798
fundamentAjusteDireto       93.661137
DescrAcordoQuadro           85.660974
numAcordoQuadro             85.660974
idINCM                      84.361231
nAnuncio                    84.361231
TipoAnuncio                 84.361231
tipoFimContrato             80.907844
dataFechoContrato           80.674750
justifNReducEscrContrato    68.780881
concorrentes                56.836875
linkPecasProc               51.144541
fundamentacao                0.235273
dataDecisaoAdjudicacao       0.233384
dataCelebracaoContrato       0.233384
LocalExecucao                0.167554
adjudicatarios               0.040108
objectoContrato              0.000436
CPV                          0.000145
idcontrato                   0.000000
ProcedimentoCentralizado     0.000000
dataPublicacao               0.000000
precoContratual              0.000000
tipoprocedimento             0.000000
idprocedimento               0.000000
descContrato                 0.000000
adjudicante 

In [4]:
colunas_remover = [
    "nAnuncio",
    "TipoAnuncio",
    "idINCM",
    "numAcordoQuadro",
    "dataFechoContrato",
    "tipoFimContrato",
    "observacoes",
    "fundamentAjusteDireto"
]


In [5]:
df = df.drop(columns=colunas_remover, errors='ignore')
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 688136 entries, 0 to 688135
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   idcontrato                688136 non-null  int64  
 1   tipoContrato              688136 non-null  object 
 2   idprocedimento            688136 non-null  int64  
 3   tipoprocedimento          688136 non-null  object 
 4   objectoContrato           688133 non-null  object 
 5   descContrato              688136 non-null  object 
 6   adjudicante               688136 non-null  object 
 7   adjudicatarios            687860 non-null  object 
 8   dataPublicacao            688136 non-null  object 
 9   dataCelebracaoContrato    686530 non-null  object 
 10  precoContratual           688136 non-null  float64
 11  CPV                       688135 non-null  object 
 12  prazoExecucao             688136 non-null  int64  
 13  LocalExecucao             686983 non-null  o

In [6]:
for col in df.select_dtypes("object"):
    if df[col].astype(str).str.contains("�").any():
        print("Problema encoding em:", col)
        
print(df.loc[df["adjudicatarios"].astype(str).str.contains("�", na=False), ["adjudicatarios"]].head(20))
print(df.loc[df["concorrentes"].astype(str).str.contains("�", na=False), ["concorrentes"]].head(20))



Problema encoding em: adjudicatarios
Problema encoding em: concorrentes
                                   adjudicatarios
384943  - - Maria Beatriz Nunes Sampaio�b6&#x1F;R
                                   concorrentes
384943  --Maria Beatriz Nunes Sampaio�b6&#x1F;R


In [7]:
for col in df.select_dtypes("object"):
    
    # remover caractere corrompido
    df[col] = df[col].str.replace("�", "", regex=False)
    
    # remover entidades HTML tipo &#x1F;
    df[col] = df[col].str.replace(r"&#x[0-9A-Fa-f]+;", "", regex=True)
    
    # remover caracteres invisíveis / não imprimíveis
    df[col] = df[col].str.replace(r"[^\x20-\x7EÀ-ÿ]", "", regex=True)

In [45]:
for col in df.select_dtypes(include="object"):
    if df[col].dropna().astype(str).str.startswith(" ").any() or df[col].dropna().astype(str).str.endswith(" ").any():
        print("Espaços extra em:", col)

Espaços extra em: objectoContrato
Espaços extra em: descContrato
Espaços extra em: adjudicante
Espaços extra em: adjudicatarios
Espaços extra em: CPV
Espaços extra em: fundamentacao
Espaços extra em: DescrAcordoQuadro
Espaços extra em: regime
Espaços extra em: justifNReducEscrContrato
Espaços extra em: concorrentes
Espaços extra em: linkPecasProc
Espaços extra em: Observacoes


In [46]:
for col in df.select_dtypes(include="object"):
    df[col] = df[col].astype(str).str.strip()

In [8]:
print("\nDuplicados:", df.duplicated().sum())

dup = df[df.duplicated(keep=False)]

id_exemplo = dup["idcontrato"].iloc[0]

print(df[df["idcontrato"] == id_exemplo])


Duplicados: 225478
        idcontrato           tipoContrato  idprocedimento tipoprocedimento  \
0         10400194  Aquisição de serviços         6772053  Consulta Prévia   
462658    10400194  Aquisição de serviços         6772053  Consulta Prévia   

                                          objectoContrato  \
0       Serviços de manutenção e assistência técnica d...   
462658  Serviços de manutenção e assistência técnica d...   

                                             descContrato  \
0       Serviços de manutenção e assistência técnica d...   
462658  Serviços de manutenção e assistência técnica d...   

                          adjudicante  \
0       506579425 - Município de Faro   
462658  506579425 - Município de Faro   

                                   adjudicatarios dataPublicacao  \
0       501445226 - TK Elevadores, Unipessoal Lda     2023-12-20   
462658  501445226 - TK Elevadores, Unipessoal Lda     2023-12-20   

       dataCelebracaoContrato  ...  dataDecisaoA

In [9]:
df = df.drop_duplicates()
print("Duplicados restantes:", df.duplicated().sum())

Duplicados restantes: 0


In [5]:

#df["idcontrato"].duplicated().sum()
#df[df["idcontrato"].duplicated(keep=False)].sort_values("idcontrato")

#df["idprocedimento"].duplicated().sum()
#df[df["idprocedimento"].duplicated(keep=False)].sort_values("idprocedimento")


(df[df["idprocedimento"].astype(str) == "1402513"])


,idcontrato,nAnuncio,TipoAnuncio,idINCM,tipoContrato,idprocedimento,tipoprocedimento,objectoContrato,descContrato,adjudicante,...,regime,justifNReducEscrContrato,tipoFimContrato,CritMateriais,concorrentes,linkPecasProc,Observacoes,ContratEcologico,fundamentAjusteDireto,Ano
8646,1397436,NaN,NaN,NaN,Aquisição de bens móveis,1402513,Ajuste Direto Regime Geral,72/0605/2015 - Aquisição de material de consum...,72/0605/2015 - Aquisição de material de consum...,"510745997 - Centro Hospitalar do Algarve, E. P...",...,Código dos Contratos Públicos (DL 18/2008),NaN,NaN,Não,NaN,NaN,NaN,Não,NaN,2015
8647,1397442,NaN,NaN,NaN,Aquisição de bens móveis,1402513,Ajuste Direto Regime Geral,72/0605/2015 - Aquisição de material de consum...,72/0605/2015 - Aquisição de material de consum...,"510745997 - Centro Hospitalar do Algarve, E. P...",...,Código dos Contratos Públicos (DL 18/2008),NaN,NaN,Não,NaN,NaN,NaN,Não,NaN,2015
8665,1397504,NaN,NaN,NaN,Aquisição de bens móveis,1402513,Ajuste Direto Regime Geral,72/0605/2015 - Aquisição de material de consum...,72/0605/2015 - Aquisição de material de consum...,"510745997 - Centro Hospitalar do Algarve, E. P...",...,Código dos Contratos Públicos (DL 18/2008),NaN,NaN,Não,NaN,NaN,NaN,Não,NaN,2015
8666,1397515,NaN,NaN,NaN,Aquisição de bens móveis,1402513,Ajuste Direto Regime Geral,72/0605/2015 - Aquisição de material de consum...,72/0605/2015 - Aquisição de material de consum...,"510745997 - Centro Hospitalar do Algarve, E. P...",...,Código dos Contratos Públicos (DL 18/2008),NaN,NaN,Não,NaN,NaN,NaN,Não,NaN,2015
8678,1397554,NaN,NaN,NaN,Aquisição de bens móveis,1402513,Ajuste Direto Regime Geral,72/0605/2015 - Aquisição de material de consum...,72/0605/2015 - Aquisição de material de consum...,"510745997 - Centro Hospitalar do Algarve, E. P...",...,Código dos Contratos Públicos (DL 18/2008),NaN,NaN,Não,NaN,NaN,NaN,Não,NaN,2015
8697,1397673,NaN,NaN,NaN,Aquisição de bens móveis,1402513,Ajuste Direto Regime Geral,72/0605/2015 - Aquisição de material de consum...,72/0605/2015 - Aquisição de material de consum...,"510745997 - Centro Hospitalar do Algarve, E. P...",...,Código dos Contratos Públicos (DL 18/2008),NaN,NaN,Não,NaN,NaN,NaN,Não,NaN,2015
8846,1397457,NaN,NaN,NaN,Aquisição de bens móveis,1402513,Ajuste Direto Regime Geral,72/0605/2015 - Aquisição de material de consum...,72/0605/2015 - Aquisição de material de consum...,"510745997 - Centro Hospitalar do Algarve, E. P...",...,Código dos Contratos Públicos (DL 18/2008),NaN,NaN,Não,NaN,NaN,NaN,Não,NaN,2015
8847,1397461,NaN,NaN,NaN,Aquisição de bens móveis,1402513,Ajuste Direto Regime Geral,72/0605/2015 - Aquisição de material de consum...,72/0605/2015 - Aquisição de material de consum...,"510745997 - Centro Hospitalar do Algarve, E. P...",...,Código dos Contratos Públicos (DL 18/2008),NaN,NaN,Não,NaN,NaN,NaN,Não,NaN,2015
8849,1397483,NaN,NaN,NaN,Aquisição de bens móveis,1402513,Ajuste Direto Regime Geral,72/0605/2015 - Aquisição de material de consum...,72/0605/2015 - Aquisição de material de consum...,"510745997 - Centro Hospitalar do Algarve, E. P...",...,Código dos Contratos Públicos (DL 18/2008),NaN,NaN,Não,NaN,NaN,NaN,Não,NaN,2015
8868,1397500,NaN,NaN,NaN,Aquisição de bens móveis,1402513,Ajuste Direto Regime Geral,72/0605/2015 - Aquisição de material de consum...,72/0605/2015 - Aquisição de material de consum...,"510745997 - Centro Hospitalar do Algarve, E. P...",...,Código dos Contratos Públicos (DL 18/2008),NaN,NaN,Não,NaN,NaN,NaN,Não,NaN,2015


ver categorias das variaveis


In [13]:
for col in df.select_dtypes(include="object"):
    print("\nColuna:", col)
    print(df[col].value_counts(dropna=False))


Coluna: tipoContrato
tipoContrato
Aquisição de bens móveis                                                          239473
Aquisição de serviços                                                             183795
Empreitadas de obras públicas                                                      29484
Locação de bens móveis                                                              5655
Aquisição de bens móveisAquisição de serviços                                       2860
Aquisição de serviçosLocação de bens móveis                                          387
Concessão de serviços públicos                                                       326
Concessão de obras públicas                                                          155
Aquisição de bens móveisLocação de bens móveis                                       125
Aquisição de serviçosEmpreitadas de obras públicas                                    99
Outros                                                                     

In [10]:
df_2015.nunique().sort_values()

linkPecasProc                    0
CritMateriais                    1
Ano                              1
ProcedimentoCentralizado         2
ContratEcologico                 2
fundamentAjusteDireto            2
TipoAnuncio                      3
tipoFimContrato                  6
tipoprocedimento                 6
justifNReducEscrContrato         7
regime                          16
tipoContrato                    32
fundamentacao                   83
DescrAcordoQuadro              286
numAcordoQuadro                297
dataCelebracaoContrato         363
dataDecisaoAdjudicacao         598
prazoExecucao                  932
dataPublicacao                1215
dataFechoContrato             1604
LocalExecucao                 1973
Observacoes                   2154
adjudicante                   4426
CPV                           5086
nAnuncio                      5364
idINCM                        5512
concorrentes                 20845
PrecoTotalEfetivo            33109
precoBaseProcediment

In [ ]:
for col in df.select_dtypes(include="object"):
    if col in df_2015.columns:
        cat1 = set(df[col].dropna().unique())
        cat2 = set(df_2015[col].dropna().unique())
        
        if cat1 != cat2:
            print("\nDiferenças na coluna:", col)
            print("Só no df1:", cat1 - cat2)
            print("Só no df2:", cat2 - cat1)


Diferenças na coluna: tipoContrato
Só no df1: {'Aquisição de bens móveisOutros', 'Aquisição de bens móveisAquisição de serviçosEmpreitadas de obras públicas', 'Aquisição de bens móveisEmpreitadas de obras públicasLocação de bens móveis', 'Aquisição de bens móveisSociedade', 'Aquisição de serviçosLocação de bens móveisSociedade', 'Aquisição de serviçosEmpreitadas de obras públicasLocação de bens móveis', 'Aquisição de bens móveisConcessão de obras públicas', 'Concessão de obras públicasConcessão de serviços públicos', 'Aquisição de bens móveisAquisição de serviços', 'Aquisição de serviçosOutrosSociedade', 'Aquisição de serviçosConcessão de obras públicas', 'Aquisição de serviçosConcessão de obras públicasConcessão de serviços públicos', 'Empreitadas de obras públicasLocação de bens móveis', 'Aquisição de bens móveisConcessão de serviços públicos', 'Aquisição de bens móveisOutrosSociedade', 'Aquisição de bens móveisAquisição de serviçosLocação de bens móveis', 'Aquisição de bens móveisC